# Step 06b — LLM Classification (benchmark arm)

**Input:** the FROZEN condensed buildings file (`config.VALIDATION_BUILDINGS_FILE`)
**Output:** `data/validation/06b_llm_predictions_{arm}_run{n}.parquet`

This notebook classifies the **1,391 annotated buildings** with the LLM so that
notebook 10 can score it against the same ground truth, through the same code, as the
rule engine.

## Why this reads the frozen file and not notebook 05's output

Notebook 05 ends with `gdf['gml_id'] = gdf.index` — the id is a **positional row
index**, assigned after a label filter and after a concat whose length depends on a
spatial join. Byte-identical code has produced 578,080 / 574,435 / 567,961 rows on
different runs, so `gml_id` 236928 is a *different building* every time.

Measured against the annotation workbook:

| classifier input | ids that resolve | volume agreement |
|---|---|---|
| regenerated notebook 05 output | 1,312 / 1,391 (94.3%) | **0 of 1,312** |
| `VALIDATION_BUILDINGS_FILE` (frozen) | 1,391 / 1,391 | **100.00%** |

A regenerated file joins cleanly and compares unrelated buildings. That is why running
the same preprocessing code on both arms does **not** give the two arms the same data —
reading the same *file* does. The rule engine's published numbers were obtained exactly
this way: notebook 10 never opens `06_classified_buildings.gpkg`, it re-runs
`classify_building` in-process over this same frozen file. The arms share the function,
never a file.

## Two evidence arms

| `ARM` | fields sent | question it answers |
|---|---|---|
| `blind` | 13 of 16 — no `osm_names`, `website`, `email` | **LLM vs rules.** `rule_utils` says business names are *"deliberately not used … the accepted limitation being measured here"*. |
| `full` | all 16 | **deployment.** What the LLM does with everything available. |

75.9% of scoreable buildings carry a name, so the full arm on its own measures
*with names vs without names*, not *LLM vs rules*. Run both; the delta prices the
world knowledge.

## Cost

One building per request, stateless. ~15–45 s per call at `LLM_MAX_WORKERS=4`, so one
run of 1,391 rows is **1.4–4.3 h**. Errors never enter the checkpoint, so re-running
this notebook *is* the retry mechanism. Drive errors to zero — the rule engine has no
exclusions (882/882 and 874/874), and any LLM row left failing shrinks its denominator
and breaks comparability.

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import (
    VALIDATION_BUILDINGS_FILE, VALIDATION_GROUND_TRUTH,
    llm_validation_checkpoint, llm_validation_errors,
    LLM_MODEL, LLM_MAX_WORKERS, LLM_CHUNK_SIZE, TARGET_MID_LABELS,
)
from llm_utils import SYSTEM_PROMPT, row_to_llm_input, predict_row

import pandas as pd
import geopandas as gpd
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Parameters ────────────────────────────────────────────────────────────────
ARM = 'blind'   # 'blind' = method comparison (no business names) | 'full' = deployment
RUN = 1         # repeat index; the LLM is not deterministic and nothing downstream
                # measures that, so >1 run is how the arm gets an error bar

CHECKPOINT = llm_validation_checkpoint(ARM, RUN)
ERRORS     = llm_validation_errors(ARM, RUN)

for path, what in [(VALIDATION_BUILDINGS_FILE, 'the frozen benchmark buildings'),
                   (VALIDATION_GROUND_TRUTH, 'notebook 09 output')]:
    if not path.exists():
        raise FileNotFoundError(f'{path}\n  missing: {what}')

print(f'arm={ARM}  run={RUN}  model={LLM_MODEL}  workers={LLM_MAX_WORKERS}')
print(f'checkpoint -> {CHECKPOINT.name}')
print(f'errors     -> {ERRORS.name}')
print(f'\nprompt: {len(SYSTEM_PROMPT):,} chars')

---
## Step 1 — Load the annotated buildings, and only those

The filter runs **before** the sentence-building `.apply`: `read_file` loads all 578,080
rows and rendering a prompt for every one of them wastes minutes for nothing.

The assertion on 1,391 is the real check that this is the right input file. A
regenerated file would drop ~79 ids here rather than silently classifying the wrong
buildings.

In [ ]:
truth = pd.read_parquet(VALIDATION_GROUND_TRUTH)
ids = set(truth['gml_id'].astype(str))
print(f'{len(ids):,} annotated building ids')

pois = gpd.read_file(VALIDATION_BUILDINGS_FILE)
print(f'{len(pois):,} buildings in {VALIDATION_BUILDINGS_FILE.name}')

pois['gml_id'] = pois['gml_id'].astype(str)
pois = pois[pois['gml_id'].isin(ids)].copy()

assert len(pois) == len(ids) == 1391, (
    f'expected 1,391 annotated buildings, matched {len(pois):,}. '
    'gml_id is a per-run positional index — this is almost certainly a REGENERATED '
    'condensed file rather than the frozen one the workbook indexes into. Scoring it '
    'would compare different buildings while every join looks healthy.')
assert pois['gml_id'].is_unique, 'gml_id not unique in the source'

# Volume agreement proves the ids point at the SAME buildings, not merely at ids that
# happen to exist. This is the check that fails loudly on a regenerated file.
chk = truth.assign(gml_id=truth['gml_id'].astype(str)).merge(
    pois[['gml_id', 'volume_m3']], on='gml_id', how='inner', suffixes=('_truth', '_src'))
rel = ((chk['volume_m3_truth'] - chk['volume_m3_src']).abs()
       / chk['volume_m3_truth'].abs().clip(lower=1e-9))
share = float((rel < 1e-3).mean())
print(f'volume agreement with the workbook: {share:.2%}')
assert share > 0.95, (
    f'only {share:.1%} of volumes agree — the ids resolve but describe different '
    'buildings. Refusing to spend API calls on this input.')

pois['sentence'] = pois.apply(lambda r: row_to_llm_input(r, fields=ARM), axis=1)
empty = int((pois['sentence'].str.len() == 0).sum())
assert empty == 0, f'{empty} buildings rendered an EMPTY prompt — the model would be asked nothing'

print(f'\nprompts built (fields={ARM}); median length '
      f'{int(pois["sentence"].str.len().median())} chars')
print('\nexample:\n')
print(pois['sentence'].iloc[0])

---
## Step 2 — Classify, checkpointing as we go

`append_parquet` rewrites the accumulated file on every flush, so an interrupted run
loses at most one chunk. Successes go to the checkpoint, failures to a separate errors
file and **never** to the checkpoint — which is what makes re-running this cell the
retry mechanism.

Every prediction is stamped with `src_file` and `src_volume_m3`. Notebook 10 asserts on
them; without the stamp its identity guard compares a value it re-joined from the source
against itself, which is bit-identical by construction and proves nothing.

**Fail fast:** if the first chunk errors above 5%, the run aborts. A systematic fault
(bad token, changed response shape, wrong argument arity) then costs 50 calls instead of
1,391 and several hours.

In [ ]:
done_ids = set()
if CHECKPOINT.exists():
    done_ids = set(pd.read_parquet(CHECKPOINT)['gml_id'].astype(str))
    print(f'Resuming: {len(done_ids):,} already done')

todo = pois[~pois['gml_id'].isin(done_ids)].copy()
print(f'Remaining to classify: {len(todo):,}')


def process_chunk(chunk_df):
    results = []
    with ThreadPoolExecutor(max_workers=LLM_MAX_WORKERS) as executor:
        futures = [
            executor.submit(predict_row, row['gml_id'], row['sentence'], ARM,
                            VALIDATION_BUILDINGS_FILE.name, row['volume_m3'])
            for _, row in chunk_df.iterrows()
        ]
        for future in as_completed(futures):
            results.append(future.result())
    return results


def append_parquet(path, new_rows):
    new_df = pd.DataFrame(new_rows)
    if path.exists():
        combined = pd.concat([pd.read_parquet(path), new_df], ignore_index=True)
        combined = combined.drop_duplicates(subset='gml_id', keep='last')
    else:
        combined = new_df
    path.parent.mkdir(parents=True, exist_ok=True)
    combined.to_parquet(path, index=False)


chunks = [todo.iloc[i:i + LLM_CHUNK_SIZE] for i in range(0, len(todo), LLM_CHUNK_SIZE)]
total_done = len(done_ids)

for i, chunk in enumerate(chunks):
    results = process_chunk(chunk)
    good   = [r for r in results if r['error'] is None]
    errors = [r for r in results if r['error'] is not None]

    if good:   append_parquet(CHECKPOINT, good)
    if errors: append_parquet(ERRORS, errors)

    total_done += len(results)
    print(f'Chunk {i+1}/{len(chunks)} | done={total_done}/{len(pois)} | errors={len(errors)}')

    if i == 0 and len(errors) > 0.05 * len(chunk):
        for e in errors[:3]:
            print(f'    {e["error"]}')
        raise RuntimeError(
            f'{len(errors)}/{len(chunk)} failed in the FIRST chunk. Aborting rather than '
            'burning hours on a systematic fault. Check the token, the endpoint and the '
            'response shape, then re-run — completed rows are already checkpointed.')

print('\nClassification pass complete.')

---
## Step 3 — Coverage

The rule engine excludes nothing. Any LLM row still failing here shrinks the LLM's
denominator, and two arms scored over different building sets are not comparable.

Re-run Step 2 until this reports zero. If a row fails repeatedly, report the residual
count alongside the metrics rather than quietly scoring 1,388 buildings against the rule
engine's 1,391.

In [ ]:
n_done = 0
if CHECKPOINT.exists():
    ck = pd.read_parquet(CHECKPOINT)
    n_done = len(ck)
    assert ck['gml_id'].is_unique, 'duplicate gml_id in the checkpoint'
    assert (ck['src_file'] == VALIDATION_BUILDINGS_FILE.name).all(), \
        'checkpoint mixes predictions derived from different source files'

n_err = 0
if ERRORS.exists():
    err = pd.read_parquet(ERRORS)
    err = err[~err['gml_id'].astype(str).isin(ck['gml_id'].astype(str))] if n_done else err
    n_err = len(err)

print(f'classified : {n_done:,} / {len(pois):,}')
print(f'outstanding: {n_err:,}')

if n_err:
    print('\nFailure modes:')
    print(err['error'].str.slice(0, 90).value_counts().head(10).to_string())
    print('\nRe-run Step 2 — errors are not checkpointed, so it retries exactly these.')
else:
    print('\nFull coverage. Comparable with the rule engine denominator.')

if n_done:
    print('\nLabel distribution:')
    print(ck['mid_labels'].explode().value_counts().to_string())
    print(f'\nnull bosserhof_class: {int(ck["bosserhof_class"].isna().sum()):,}')

---
## Done

Score it with **notebook 10**, which discovers every `06b_llm_predictions_*.parquet`
and scores each as its own arm against the same ground truth, through the same
`validation_utils` calls, as the rule engine.

To produce the other arms, set `ARM`/`RUN` at the top and re-run:

| `ARM` | `RUN` | purpose |
|---|---|---|
| `blind` | 1, 2, 3 | primary comparison + run-to-run variance |
| `full` | 1 | value of business names |

---

### Production run (not part of the benchmark — leave unrun)

Classifying the full `CONDENSED_BUILDINGS_FILE` writes `LLM_CHECKPOINT_FILE`, feeds
notebook 06c, and produces zone-level results comparable to `08_final_results.gpkg`.

At 15–45 s/call and 4 workers, 578,080 buildings is **25–75 days of wall clock**. It
answers no question the 1,391-row benchmark does not already answer. The cell below is
deliberately left unexecuted.

In [ ]:
# PRODUCTION RUN — 578,080 calls, 25-75 days. Do not run casually.
#
# Uses LLM_CHECKPOINT_FILE, deliberately separate from the benchmark checkpoints: the
# frozen and regenerated files draw gml_ids from the same small-integer namespace while
# describing different buildings, so a shared checkpoint would interleave predictions for
# unrelated buildings and nothing downstream could detect it.
#
# from config import CONDENSED_BUILDINGS_FILE, LLM_CHECKPOINT_FILE, LLM_ERRORS_FILE
# prod = gpd.read_file(CONDENSED_BUILDINGS_FILE)
# prod['sentence'] = prod.apply(lambda r: row_to_llm_input(r, fields='full'), axis=1)
# ... same chunk loop, writing LLM_CHECKPOINT_FILE / LLM_ERRORS_FILE ...
pass